## Manual Review of LLM Scope/Pillar Classifications (S5)

Turns `scope_LLM` / `confidence_LLM` / `pillar_LLM` (from `S3_LLM_scope.py`) into a trusted `scope_curated` / `pillar_curated` verdict on `funding_classified`. LLM output alone isn't ground truth - confident decisions are auto-inherited, everything else goes to a human.

Standalone notebook, not invoked by `pipeline_funding.py` - run it whenever there's new LLM-scored data to review, independent of any single pipeline run.

Auto-accept logic (no ML stage in Funding, so this is a plain confidence-band split, unlike Publications' confidence+ML-pillar-agreement rule):
- Confidence 5-7 with a real pillar assigned -> auto-approve (`scope_curated='in'`)
- Confidence 1 with no pillar assigned -> auto-reject (`scope_curated='out'`) - "reject" means excluded from being promoted onward, **not** deleted from `funding_classified` (deleting it would break S1's re-query dedup check)
- Confidence 2-4 -> manual review
- Anything with a missing/inconsistent scope, pillar, or LLM status also falls through to manual review, regardless of confidence

In [2]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

DB_PATH = Path('funding.db')  # self-contained in Pipeline/Funding, mirrors Publications
DATA_DIR = Path('data_review')
DATA_DIR.mkdir(exist_ok=True)

today = datetime.today().strftime("%y%m%d")
RUN_TIMESTAMP = datetime.today().strftime("%y%m%d_%H%M")  # for filenames, so reruns don't overwrite

## Pull LLM-scored grants for manual review

In [2]:
# Connect to database
db = duckdb.connect(str(DB_PATH))

In [3]:
# Show tables in database
db.sql("SHOW TABLES")

┌───────────────────────┐
│         name          │
│        varchar        │
├───────────────────────┤
│ funding_classified    │
│ funding_curated       │
│ run_260727_0922       │
│ run_260727_0922_dedup │
└───────────────────────┘

#### Get data from `funding_classified`

In [4]:
data = db.sql("SELECT * FROM funding_classified").df()

In [5]:
# Only rows that have actually been through S3 (LLM scoring)
data = data[data['status_LLM'].notna()]

In [6]:
# funding_classified grows forever, so re-running this notebook must not re-export rows
# already curated in an earlier pass - only look at rows with no scope_curated yet.
if 'scope_curated' in data.columns:
    data = data[data['scope_curated'].isna()]

print(f"{len(data)} LLM-scored grants awaiting curation")

4177 LLM-scored grants awaiting curation


#### Create filter mask for curated scope and pillar

In [7]:
# decision mask for auto-accept vs manual review
auto_in_mask = (
    (data['status_LLM'] == 'ok') &
    (data['scope_LLM'] == 'in') &
    (data['confidence_LLM'] >= 5) &
    (data['pillar_LLM'].notna()) & (data['pillar_LLM'] != 'NA')
)

auto_out_mask = (
    (data['status_LLM'] == 'ok') &
    (data['scope_LLM'] == 'out') &
    (data['confidence_LLM'] == 1) &
    (data['pillar_LLM'] == 'NA')
)

inherit_mask = auto_in_mask | auto_out_mask

#### Assign curated scope/pillar or manual review

In [8]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated']  = np.where(inherit_mask, data['scope_LLM'],  'manual_review')
data['pillar_curated'] = np.where(inherit_mask, data['pillar_LLM'], 'manual_review')
data['date_review'] = today

In [9]:
# How many grants have to go through manual review?
data['scope_curated'].value_counts()

scope_curated
out              2791
manual_review     866
in                520
Name: count, dtype: int64

#### Save borderline grants for manual review

In [10]:
# Export grants for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(DATA_DIR / f'{RUN_TIMESTAMP}_funding_for_review.csv', index=False)
print(f"Saved {len(review)} grants for review -> {DATA_DIR / f'{RUN_TIMESTAMP}_funding_for_review.csv'}")

Saved 866 grants for review -> data_review\260729_2124_funding_for_review.csv


#### Save (automatically) curated scope and pillar back to `funding_classified`

In [11]:
# filter for the grants with a clear scope and pillar assignment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [12]:
# create columns in funding_classified if they don't already exist
db.sql("ALTER TABLE funding_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql("ALTER TABLE funding_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [13]:
# add to database
db.register('data', data[['Grant ID', 'scope_curated', 'pillar_curated']])
db.sql("""
    UPDATE funding_classified
    SET scope_curated  = data.scope_curated,
        pillar_curated = data.pillar_curated
    FROM data
    WHERE funding_classified."Grant ID" = data."Grant ID"
""")
print(f"Auto-curated {len(data)} grants in funding_classified")

Auto-curated 3311 grants in funding_classified


In [14]:
db.close()

## Assign manually curated scope and pillar to respective grants

Run this section after you've opened the exported CSV, filled in real `scope_curated` / `pillar_curated` values (replacing `manual_review`) for each row, and saved it as `{timestamp}_funding_reviewed.csv` in the same `data/` folder.

In [3]:
# Connect to database
db = duckdb.connect(str(DB_PATH))

In [4]:
# load manually reviewed data
### MUST UPDATE the date in this filename to match your reviewed export
# keep_default_na=False + na_values=[''] - pandas' default NA-string list includes the literal
# text 'NA', so without this, every out-of-scope row's pillar_curated='NA' silently becomes a
# null on load even though it's typed correctly in the CSV. Only a truly empty cell should
# read as missing.
reviewed_data = pd.read_csv(
    DATA_DIR / f'260729_2124_funding_reviewed.csv',  # <-- set to match the reviewed file you saved
    keep_default_na=False,
    na_values=[''],
)

#### Add scope and pillar back to `funding_classified`, using grant ID

In [5]:
# add to database
db.register('reviewed_data', reviewed_data[['Grant ID', 'scope_curated', 'pillar_curated']])
db.sql("""
    UPDATE funding_classified
    SET scope_curated  = reviewed_data.scope_curated,
        pillar_curated = reviewed_data.pillar_curated
    FROM reviewed_data
    WHERE funding_classified."Grant ID" = reviewed_data."Grant ID"
""")
print(f"Applied manual review decisions for {len(reviewed_data)} grants")

Applied manual review decisions for 866 grants


In [6]:
# Validate & auto-fix scope_curated / pillar_curated across the whole of funding_classified.
# Auto-fixes (safe, mechanical): strips stray whitespace (e.g. a typo'd 'out '); backfills
# pillar_curated='NA' for any out-of-scope row that's missing one - matches the convention
# auto-curation already uses, and acts as a backstop for the read_csv issue fixed above.
# Everything else is reported, not guessed at - e.g. an in-scope grant with no real pillar
# needs a human, not code. Runs directly against the live table, so it's safe to re-run any
# time (including without re-running the cells above) and won't touch unaffected rows.
VALID_PILLARS = {'PB', 'F', 'CM', 'CC'}

current = db.sql('SELECT "Grant ID", scope_curated, pillar_curated FROM funding_classified WHERE scope_curated IS NOT NULL').df()

fixed = current.copy()
fixed['scope_curated'] = fixed['scope_curated'].str.strip()
fixed['pillar_curated'] = fixed['pillar_curated'].str.strip()
needs_na_fill = (fixed['scope_curated'] == 'out') & fixed['pillar_curated'].isna()
fixed.loc[needs_na_fill, 'pillar_curated'] = 'NA'


def _differs(a, b):
    return ~((a == b) | (a.isna() & b.isna()))


changed = fixed[_differs(fixed['scope_curated'], current['scope_curated']) | _differs(fixed['pillar_curated'], current['pillar_curated'])]

if len(changed):
    db.register('fixed_rows', changed[['Grant ID', 'scope_curated', 'pillar_curated']])
    db.sql("""
        UPDATE funding_classified
        SET scope_curated  = fixed_rows.scope_curated,
            pillar_curated = fixed_rows.pillar_curated
        FROM fixed_rows
        WHERE funding_classified."Grant ID" = fixed_rows."Grant ID"
    """)
    db.unregister('fixed_rows')
    print(f"Auto-fixed {len(changed)} row(s): trimmed whitespace and/or backfilled 'NA' pillar for out-of-scope grants.")
else:
    print("No auto-fixable formatting issues found.")

# Re-check post-fix and flag anything that still needs a human
current = db.sql('SELECT "Grant ID", scope_curated, pillar_curated FROM funding_classified WHERE scope_curated IS NOT NULL').df()

bad_scope      = current[~current['scope_curated'].isin(['in', 'out', 'manual_review'])]
bad_pillar_in  = current[(current['scope_curated'] == 'in') & (~current['pillar_curated'].isin(VALID_PILLARS))]
bad_pillar_out = current[(current['scope_curated'] == 'out') & (current['pillar_curated'] != 'NA')]
still_manual   = current[current['scope_curated'] == 'manual_review']

for label, subset in [
    ("invalid scope_curated value", bad_scope),
    (f"in-scope but pillar_curated not in {VALID_PILLARS}", bad_pillar_in),
    ("out-of-scope but pillar_curated != 'NA'", bad_pillar_out),
    ("still stuck at 'manual_review'", still_manual),
]:
    if len(subset):
        print(f"\n[ISSUE] {len(subset)} row(s) - {label}:")
        print(subset.to_string(index=False))

if not any(len(x) for x in (bad_scope, bad_pillar_in, bad_pillar_out, still_manual)):
    print("\nAll clear - scope_curated / pillar_curated are complete and correctly formatted.")

Auto-fixed 2 row(s): trimmed whitespace and/or backfilled 'NA' pillar for out-of-scope grants.

All clear - scope_curated / pillar_curated are complete and correctly formatted.


In [7]:
data = db.sql("SELECT * FROM funding_classified").df()
data[~data['scope_curated'].isna()]

,Grant ID,Title translated,Title,Abstract translated,Abstract,Researchers,Research Organization - standardized,Currency,Total amount,Total amount (USD),...,pillar_LLM,plant_based_LLM,fermentation_LLM,cultivated_LLM,cross_cutting_LLM,status_LLM,stop_reason_LLM,date_LLM,scope_curated,pillar_curated
0,grant.15151354,3D Human Trigeminal Ganglion Organoids: The Et...,3D Human Trigeminal Ganglion Organoids: The Et...,Pain research is one of the most ethically com...,<NA>,Pierfrancesco Pagella,Linköping University,SEK,NaN,243508.0,...,NA,False,False,False,False,ok,tool_use,260729,out,NA
1,grant.15468476,First development action for Medical Counter M...,First development action for Medical Counter M...,The European Union (EU) faces growing security...,<NA>,NaN,Commissariat à l'Énergie Atomique et Aux Énerg...,EUR,9803529.0,11433767.0,...,NA,False,False,False,False,ok,tool_use,260729,out,NA
2,grant.15280123,Co-fermentation of pea protein isolate with Ba...,Co-Fermentation von Erbsenproteinisolat mit Ba...,NaN,<NA>,NaN,Research Association of the German Food Industry,EUR,524504.0,611482.0,...,PB,True,False,False,False,ok,tool_use,260729,in,PB
3,grant.15118405,Evaluation of the functional properties of xyl...,Avaliação das propriedades funcionais de xilo-...,Xylo-oligosaccharides (XOS) are non-digestible...,<NA>,Rosana Coelho; Marina de Lima Tresmondi,State University of Campinas,NaN,NaN,NaN,...,PB,True,False,False,False,ok,tool_use,260729,in,PB
4,grant.15289209,SME Innovative - SMEi-BÖ09: RheoTexMeat: Rheol...,KMU-innovativ -KMUi-BÖ09: RheoTexMeat: Rheolog...,NaN,<NA>,NaN,TU Dresden,EUR,610391.0,711206.0,...,CC,False,False,False,True,ok,tool_use,260729,in,CC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4172,grant.14846989,Improving fruit firmness: Editing genes involv...,MEJORA DE LA FIRMEZA DEL FRUTO: EDICION DE GEN...,Strawberries are characterized by excessive so...,<NA>,NaN,University of Malaga,EUR,269920.0,307671.0,...,NA,False,False,False,False,ok,tool_use,260729,out,NA
4173,grant.14846569,New sustainable aquaculture technologies for o...,NUEVAS TECNOLOGIAS ACUICOLAS SOSTENIBLES PARA ...,The AQUATECH4FEED project aims to develop a no...,<NA>,NaN,University of Almería,EUR,150000.0,170979.0,...,NA,False,False,False,False,ok,tool_use,260729,out,NA
4174,grant.14066368,Valorisation of EPS (extracellular polymeric s...,Valorisation of EPS (extracellular polymeric s...,"During the waste water cleaning process , a ne...",<NA>,Jurgen Joossens,University of Antwerp,NaN,NaN,NaN,...,NA,False,False,False,False,ok,tool_use,260729,out,NA
4175,grant.13690862,Discerning the synergism of mixed-type plant-b...,Discerning the synergism of mixed-type plant-b...,Lutein’s poor water-solubility necessitates th...,<NA>,Tan Boon,Universiti Putra Malaysia,NaN,NaN,NaN,...,NA,False,False,False,False,ok,tool_use,260729,out,NA


In [8]:
db.close()